In [2]:
import duckdb as ddb
import pandas as pd

In [3]:
con = ddb.connect("../air_quality.db")
con

In [4]:
con.sql("SHOW TABLES")
con.sql("SELECT COUNT(*) FROM raw.air_quality_data")


┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        75128 │
└──────────────┘

In [12]:
df = con.query("select * from raw.air_quality_data where parameter in ('o3', 'no2', 'pm25', 'pm10','so2')").to_df()
df.tail()

,location_id,sensors_id,location,datetime,lat,lon,parameter,units,value,month,year,ingestion_datetime
50110,921005,5077601,Downtown Vancouver-931741,2025-05-31 20:00:00,49.282,-123.122,o3,ppm,0.028,05,2025,2025-07-08 11:57:41.612
50111,921005,5077601,Downtown Vancouver-931741,2025-05-31 21:00:00,49.282,-123.122,o3,ppm,0.026,05,2025,2025-07-08 11:57:41.612
50112,921005,5077601,Downtown Vancouver-931741,2025-05-31 22:00:00,49.282,-123.122,o3,ppm,0.030,05,2025,2025-07-08 11:57:41.612
50113,921005,5077601,Downtown Vancouver-931741,2025-05-31 23:00:00,49.282,-123.122,o3,ppm,0.029,05,2025,2025-07-08 11:57:41.612
50114,921005,5077601,Downtown Vancouver-931741,2025-06-01 00:00:00,49.282,-123.122,o3,ppm,0.030,05,2025,2025-07-08 11:57:41.612


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50115 entries, 0 to 50114
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   location_id         50115 non-null  int64         
 1   sensors_id          50115 non-null  int64         
 2   location            50115 non-null  object        
 3   datetime            50115 non-null  datetime64[us]
 4   lat                 50115 non-null  float64       
 5   lon                 50115 non-null  float64       
 6   parameter           50115 non-null  object        
 7   units               50115 non-null  object        
 8   value               50115 non-null  float64       
 9   month               50115 non-null  object        
 10  year                50115 non-null  int64         
 11  ingestion_datetime  50115 non-null  datetime64[us]
dtypes: datetime64[us](2), float64(3), int64(3), object(4)
memory usage: 4.6+ MB


In [14]:
df.describe()

,location_id,sensors_id,datetime,lat,lon,value,year,ingestion_datetime
count,50115.000000,5.011500e+04,50115,50115.000000,50115.000000,50115.000000,50115.0,50115
mean,307854.471735,2.192761e+06,2025-03-17 17:00:33.259503,49.235662,-123.055014,0.636798,2025.0,2025-07-08 11:54:58.074265
min,1528.000000,2.691000e+03,2025-01-01 01:00:00,49.186390,-123.152220,-0.002000,2025.0,2025-07-08 11:52:04.101000
25%,1528.000000,2.203400e+04,2025-02-07 11:00:00,49.186390,-123.152220,0.000800,2025.0,2025-07-08 11:53:37.181000
50%,1600.000000,1.714583e+06,2025-03-18 01:00:00,49.215280,-123.122000,0.010000,2025.0,2025-07-08 11:54:17.962000
75%,921003.000000,5.077601e+06,2025-04-24 13:00:00,49.282000,-122.985560,0.030000,2025.0,2025-07-08 11:56:44.981000
max,921005.000000,5.077843e+06,2025-06-01 00:00:00,49.287000,-122.922300,39.100000,2025.0,2025-07-08 11:57:41.612000
std,401322.250415,1.957590e+06,NaN,0.041702,0.087823,1.847822,0.0,NaN


In [15]:
df.describe(include = 'O')

,location,parameter,units,month
count,50115,50115,50115,50115
unique,5,4,2,5
top,Vancouver Airport-1528,o3,ppm,01
freq,14319,14328,42962,10366


In [16]:
df.parameter.unique()

array(['pm25', 'no2', 'so2', 'o3'], dtype=object)

In [17]:
df[df.duplicated(subset=["location_id", "parameter", "units", "value", "datetime"])]

,location_id,sensors_id,location,datetime,lat,lon,parameter,units,value,month,year,ingestion_datetime


In [18]:
df.groupby(by = "parameter", as_index = False).count()

,parameter,location_id,sensors_id,location,datetime,lat,lon,units,value,month,year,ingestion_datetime
0,no2,14318,14318,14318,14318,14318,14318,14318,14318,14318,14318,14318
1,o3,14328,14328,14328,14328,14328,14328,14328,14328,14328,14328,14328
2,pm25,7153,7153,7153,7153,7153,7153,7153,7153,7153,7153,7153
3,so2,14316,14316,14316,14316,14316,14316,14316,14316,14316,14316,14316


In [19]:
df[df.value < 0 ]["location"].unique()

array(['Downtown Vancouver-931741'], dtype=object)

In [26]:
con.sql("SELECT *  FROM presentation.air_quality_data where location = 'Burnaby North-931739' AND value is null")

┌─────────────┬────────────┬──────────┬───────────┬────────┬────────┬───────────┬─────────┬────────┬─────────┬───────┬────────────────────┐
│ location_id │ sensors_id │ location │ datetime  │  lat   │  lon   │ parameter │  units  │ value  │  month  │ year  │ ingestion_datetime │
│    int64    │   int64    │ varchar  │ timestamp │ double │ double │  varchar  │ varchar │ double │ varchar │ int64 │     timestamp      │
├─────────────┴────────────┴──────────┴───────────┴────────┴────────┴───────────┴─────────┴────────┴─────────┴───────┴────────────────────┤
│                                                                 0 rows                                                                  │
└─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┘

In [27]:
con.close()